In [1]:
import os
import torch
import numpy as np
from astropy.table import Table as aT

In [2]:
from sedflow import flows as F

/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import corner as DFM
# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams['text.usetex'] = True
#mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

In [4]:
bgs = aT.read('/global/cfs/projectdirs/desi/survey/catalogs/Y1/LSS/iron/LSScats/v1.2/BGS_ANY_full.dat.fits')
#bgs = aT.read('/global/cfs/projectdirs/desi/survey/catalogs/SV3/LSS/fuji/LSScats/EDAbeta/BGS_ANY_full.dat.fits')

In [5]:
# select only BGS objects with good spectra with successful redshifts
is_good = (
    (bgs['COADD_FIBERSTATUS'] == 0) & 
    (bgs['SPECTYPE'] == 'GALAXY') & 
    (bgs['Z_not4clus'] > 0.) & 
    (bgs['ZWARN'] == 0) & 
    (bgs['DELTACHI2'] > 40.) & 
    (bgs['ZERR'] < 0.0005 * (1 + bgs['Z_not4clus'])))

print('%i out of %i BGS z-success' % (np.sum(is_good), len(is_good)))

5785742 out of 10824424 BGS z-success


In [6]:
bgs = bgs[is_good]#[:10000]

In [7]:
alreadyrun = np.ones(len(np.arange(len(bgs))[::1000])).astype(bool)

for ii, igal in enumerate(np.arange(len(bgs))[::1000]): 
    fpost = os.path.join('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/',
                         'desiy1.bgs.sedflow.modelb.lowz.cdf.grzW1W2.%i.npy' % bgs['TARGETID'][igal])
    
    if not os.path.isfile(fpost):
        alreadyrun[ii] = False

In [8]:
for i in range(1,len(alreadyrun)-1): 
    if alreadyrun[i-1] + alreadyrun[i] + alreadyrun[i+1] == 0: 
        print(np.arange(len(bgs))[::1000][i])